<a href="https://colab.research.google.com/github/pachterlab/varseek-examples/blob/main/vk_denovo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [vk denovo](https://github.com/pachterlab/varseek) demonstration
Call de novo variants from scRNA-seq reads, build a varseek reference from those variants, and count variant-supporting reads with vk count. This notebook uses a [10x PBMC 1k dataset](https://www.10xgenomics.com/datasets/1-k-pbm-cs-from-a-healthy-donor-v-3-chemistry-3-standard-3-0-0) as an example.

Written by Joseph Rich.
___


### Install varseek, and import all packages

In [4]:
try:
    import varseek as vk
except ImportError:
    print("varseek not found, installing...")
    !pip install -U -q varseek

In [5]:
import os
import re
import anndata as ad
import numpy as np
import pandas as pd
import pysam

import varseek as vk

In [6]:
# !pip install -q ipython-autotime
%load_ext autotime

time: 121 μs (started: 2026-07-16 15:44:09 -07:00)


### Define important paths

In [7]:
# input files
fastqs_dir = os.path.join("data", "pbmc_1k_v3_fastqs")
technology = "10xv3"
reference_dir = os.path.join("data", "reference")
sequences = os.path.join(reference_dir, "ensembl_grch38_release114", "Homo_sapiens.GRCh38.dna.primary_assembly.fa")
gtf = os.path.join(reference_dir, "ensembl_grch38_release114", "Homo_sapiens.GRCh38.114.gtf")

root = os.path.join("data", "cosmic_vs_denovo")

# COSMIC census catalog (defined here so the de novo filter cell can use it)
cosmic_tsv = os.path.join("data", "cosmic", "Cosmic_MutantCensus_v104_GRCh38.tsv")

# vk denovo out -- variant calls reused from the earlier de novo run
variants_dir = os.path.join("data", "pbmc_1k_v3_variants")
variants_vcf = os.path.join(variants_dir, "variants.vcf.gz")
variants = os.path.join(variants_dir, "variants.tsv")
variants_cosmic_only = os.path.join(variants_dir, "variants_cosmic_only.tsv")
denovo_bam_dir = os.path.join(variants_dir, "bams")
star_genome_index_dir = os.path.join(reference_dir, "star_index")
denovo_star_alignment_dir = os.path.join(variants_dir, "star_alignments")

# de novo calls restricted to those that also appear in the COSMIC census. Building the
# de novo pipeline from this subset puts both pipelines on a shared variant universe, so
# the de novo VCRS reference is (by construction) a subset of the full COSMIC reference.


# vk ref and count out denovo (COSMIC-only subset; small, built by this notebook)
vk_ref_out_dir_denovo = os.path.join(root, "vk_ref_denovo")
vcrs_index_denovo = os.path.join(vk_ref_out_dir_denovo, "vcrs_index_denovo.idx")
vcrs_t2g_denovo = os.path.join(vk_ref_out_dir_denovo, "vcrs_t2g_denovo.txt")
vk_count_out_dir_denovo = os.path.join(root, "vk_count_denovo")
vk_count_variants_vcf_denovo = os.path.join(vk_count_out_dir_denovo, "varseek_variants.vcf")

# vk ref and count out cosmic
vk_ref_out_dir_cosmic = os.path.join(root, "vk_ref_cosmic")
vcrs_index_cosmic = os.path.join(vk_ref_out_dir_cosmic, "vcrs_index_cosmic.idx")
vcrs_t2g_cosmic = os.path.join(vk_ref_out_dir_cosmic, "vcrs_t2g_cosmic.txt")
vk_count_out_dir_cosmic = os.path.join(root, "vk_count_cosmic")
vk_count_variants_vcf_cosmic = os.path.join(vk_count_out_dir_cosmic, "varseek_variants.vcf")

# general parameters
w = 37
k = 41
min_counts_denovo = 3
threads = 16
min_counts_clean = 1          # per-cell vk clean threshold (kept at 1; see min_total_reads below)
min_total_reads_denovo = None
min_total_reads_cosmic = None

time: 3.76 ms (started: 2026-07-16 15:44:09 -07:00)


In [8]:
if not os.path.exists(cosmic_tsv):
    raise FileNotFoundError(f"Missing COSMIC census catalog: {cosmic_tsv}. Please download from https://cancer.sanger.ac.uk/cosmic/download and place in the data/cosmic directory.")

time: 30.3 ms (started: 2026-07-16 15:44:09 -07:00)


In [9]:
if not os.path.exists(sequences):
    sequences_dir = os.path.dirname(sequences)
    !gget ref -w dna -r 114 --out_dir {sequences_dir} -d human
    !gunzip {sequences}.gz

if not os.path.exists(gtf):
    gtf_dir = os.path.dirname(gtf)
    !gget ref -w gtf -r 114 --out_dir {gtf_dir} -d human
    !gunzip {gtf}.gz

time: 22.6 ms (started: 2026-07-16 15:44:09 -07:00)


In [10]:
def adata_to_vcf(adata, vcf_out, min_total_reads=None):
    if os.path.exists(vcf_out):
        print(f"{vcf_out} already exists, skipping VCF writing.")
        return

    if isinstance(adata, str):
        adata = ad.read_h5ad(adata)
    if not isinstance(adata, ad.AnnData):
        raise ValueError("adata must be either a file path or an AnnData object.")

    # Keep a variant only if its TOTAL supporting reads (summed across cells) reach
    # min_total_reads. vk clean's `min_counts` is a per-cell-entry threshold, which on
    # sparse 10x data (~1 read/cell) is far too aggressive -- it also deletes most true
    # calls; a threshold on the column total is the right control for calling a variant
    # from pseudobulked counts. min_total_reads=1 reproduces the original "present in
    # >=1 cell" behavior; raising it (e.g. 2 for the COSMIC screen) drops the
    # single-read-noise false positives while keeping ~3/4 of the true calls.
    if min_total_reads is not None:
        adata = adata[:, np.asarray(adata.X.sum(axis=0)).ravel() >= min_total_reads]
    variants = adata.var_names.tolist()

    # Reference genome FASTA
    fasta = pysam.FastaFile(sequences)

    chromosomes = {str(i) for i in range(1, 23)}.union({"X", "Y", "MT"})

    # Flatten grouped variants (joined by ";") into one Series, preserving the
    # original order so the written VCF matches the old per-variant loop.
    var_series = pd.Series(variants, dtype="object").str.split(";").explode()
    var_series = var_series[var_series.str.len() > 0].reset_index(drop=True)

    # Vectorized HGVS parsing: every variant is exactly one of three forms.
    # str.extract returns NaN for every group on a non-matching row, so a
    # notna() check on a required group acts as the per-type mask.
    snv = var_series.str.extract(r"^(?P<CHROM>.+):g\.(?P<POS>\d+)(?P<REF>[ACGT]+)>(?P<ALT>[ACGT]+)$")
    ins = var_series.str.extract(r"^(?P<CHROM>.+):g\.(?P<POS>\d+)_(?P<END>\d+)ins(?P<INS>[ACGT]+)$")
    dele = var_series.str.extract(r"^(?P<CHROM>.+):g\.(?P<START>\d+)(?:_(?P<END>\d+))?del(?P<DEL>[ACGT]*)$")

    snv_mask = snv["CHROM"].notna()
    ins_mask = ins["CHROM"].notna()
    del_mask = dele["START"].notna()

    for var in var_series[~(snv_mask | ins_mask | del_mask)]:
        print(f"Could not parse: {var}")

    frames = []

    # --- Substitutions (no reference lookup needed) ---
    snv = snv[snv_mask & snv["CHROM"].isin(chromosomes)]
    if not snv.empty:
        frames.append(pd.DataFrame({
            "CHROM": snv["CHROM"],
            "POS": snv["POS"].astype(int),
            "REF": snv["REF"],
            "ALT": snv["ALT"],
        }, index=snv.index))

    # --- Insertions: prepend the left anchor base from the reference ---
    ins = ins[ins_mask & ins["CHROM"].isin(chromosomes)]
    if not ins.empty:
        ins_pos = ins["POS"].astype(int)
        anchor = pd.Series(
            [fasta.fetch(c, p - 1, p) for c, p in zip(ins["CHROM"], ins_pos)],
            index=ins.index,
        )
        frames.append(pd.DataFrame({
            "CHROM": ins["CHROM"],
            "POS": ins_pos,
            "REF": anchor,
            "ALT": anchor + ins["INS"],
        }, index=ins.index))

    # --- Deletions: anchor base before the deletion; fetch the deleted bases
    #     from the reference when HGVS omits them (e.g. "g.123del") ---
    dele = dele[del_mask & dele["CHROM"].isin(chromosomes)]
    if not dele.empty:
        start = dele["START"].astype(int)
        end = pd.to_numeric(dele["END"]).fillna(start).astype(int)
        anchor = pd.Series(
            [fasta.fetch(c, s - 2, s - 1) for c, s in zip(dele["CHROM"], start)],
            index=dele.index,
        )
        deleted = pd.Series(
            [d if d != "" else fasta.fetch(c, s - 1, e)
            for c, s, e, d in zip(dele["CHROM"], start, end, dele["DEL"])],
            index=dele.index,
        )
        frames.append(pd.DataFrame({
            "CHROM": dele["CHROM"],
            "POS": start - 1,
            "REF": anchor + deleted,
            "ALT": anchor,
        }, index=dele.index))

    # Concatenate all variant types and restore the original variant order.
    vcf_df = (pd.concat(frames).sort_index()
            if frames else pd.DataFrame(columns=["CHROM", "POS", "REF", "ALT"]))

    # Build every data line with vectorized string ops, then write in one shot.
    lines = (
        vcf_df["CHROM"].astype(str) + "\t"
        + vcf_df["POS"].astype(str) + "\t.\t"
        + vcf_df["REF"] + "\t"
        + vcf_df["ALT"] + "\t.\t.\t.\n"
    )

    with open(vcf_out, "w") as f:
        f.write("##fileformat=VCFv4.2\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        f.write("".join(lines))

    print(f"Wrote {len(vcf_df):,} variants to {vcf_out}")

time: 34.9 ms (started: 2026-07-16 15:44:09 -07:00)


In [11]:
import os
import gzip
import subprocess

def is_vcf_normalized(vcf_path):
    """
    Returns True if the VCF has a header line indicating bcftools norm was run.
    """
    # Auto-detect if compressed
    open_func = gzip.open if vcf_path.endswith(".gz") else open

    with open_func(vcf_path, "rt") as f:
        for line in f:
            if line.startswith("##bcftools_normCommand="):
                return True
            # Headers always start with ##
            if not line.startswith("##"):
                # Stop at the column header line
                break
    return False

def ensure_genotype_column(vcf_path, sample_name="SAMPLE", genotype="1/1"):
    """hap.py requires at least one sample carrying genotypes. varseek emits a
    sites-only VCF (CHROM..INFO only), so inject a placeholder FORMAT/GT column
    if none is present. No-op for VCFs that already have sample columns."""
    open_func = gzip.open if vcf_path.endswith(".gz") else open

    with open_func(vcf_path, "rt") as f:
        lines = f.readlines()

    for line in lines:
        if line.startswith("#CHROM"):
            if len(line.rstrip("\n").split("\t")) > 8:
                return  # FORMAT + sample columns already present
            break

    gt_header = '##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">\n'
    new_lines = []
    inserted_header = False
    for line in lines:
        if line.startswith("##"):
            new_lines.append(line)
        elif line.startswith("#CHROM"):
            if not inserted_header:
                new_lines.append(gt_header)
                inserted_header = True
            cols = line.rstrip("\n").split("\t") + ["FORMAT", sample_name]
            new_lines.append("\t".join(cols) + "\n")
        elif line.strip():
            cols = line.rstrip("\n").split("\t") + ["GT", genotype]
            new_lines.append("\t".join(cols) + "\n")

    with open(vcf_path, "wt") as f:
        f.writelines(new_lines)

def insert_suffix_before_first_dot(path, suffix):
    dirname, basename = os.path.split(path)
    if "." in basename:
        parts = basename.split(".", 1)
        new_basename = parts[0] + suffix + "." + parts[1]
    else:
        new_basename = basename + suffix
    return os.path.join(dirname, new_basename)

def add_normalized_before_first_dot(path):
    return insert_suffix_before_first_dot(path, "_normalized")

def _vcf_has_samples(vcf_path):
    """True if the VCF carries FORMAT + per-sample genotype columns (i.e. the
    #CHROM header line has more than the 8 fixed columns)."""
    open_func = gzip.open if vcf_path.endswith(".gz") else open
    with open_func(vcf_path, "rt") as f:
        for line in f:
            if line.startswith("#CHROM"):
                return len(line.rstrip("\n").split("\t")) > 8
            if not line.startswith("#"):
                break
    return False

def filter_to_variant_calls(vcf_path):
    """Keep every record that lists a real ALT allele, regardless of genotype --
    including hom-ref (0/0) and no-call (./.) calls (e.g. bcftools/DeepVariant sites
    that still carry an alternate allele). Only gVCF reference blocks (ALT=".") are
    dropped. The FILTER column is
    preserved so hap.py can still stratify ALL vs PASS."""
    # Sites-only VCFs (no FORMAT/sample columns, e.g. varseek or VarTrix) carry no
    # genotypes, so an ALT-based filter still applies; but there is nothing to drop
    # -- every record is already a variant site -- so pass the
    # VCF through unchanged (and skip the _nonref suffix downstream).
    if not _vcf_has_samples(vcf_path):
        return vcf_path

    out_path = insert_suffix_before_first_dot(vcf_path, "_nonref")

    if os.path.isfile(out_path) and os.path.getsize(out_path) > 0:
        print(f"Filtered file {out_path} already exists. Skipping filtering.")
    else:
        output_type = "-Oz" if out_path.endswith(".vcf.gz") else "-Ov"
        subprocess.run(
            ["bcftools", "view", "-i", 'ALT!="."', output_type, "-o", out_path, vcf_path],
            check=True,
        )
    return out_path

def make_normalized_vcf(test_vcf, reference_fasta):
    test_vcf_unnormalized = test_vcf
    test_vcf = add_normalized_before_first_dot(test_vcf_unnormalized)

    # Treat a zero-byte file as missing: a previous bcftools failure can leave an
    # empty output behind, and a plain isfile() check would silently reuse it.
    if os.path.isfile(test_vcf) and os.path.getsize(test_vcf) > 0:
        print(f"Normalized file {test_vcf} already exists. Skipping normalization.")
    else:
        reference_fasta_index = f"{reference_fasta}.fai"
        if not os.path.isfile(reference_fasta_index):
            subprocess.run(["samtools", "faidx", reference_fasta], check=True)

        output_type = "-Oz" if test_vcf.endswith(".vcf.gz") else "-Ov"
        # Pipeline: reheader -> norm -> sort.
        #  * reheader --fai: the varseek VCF carries no ##contig header lines, which
        #    makes `bcftools norm` abort with "CONTIG id not present in the header"
        #    and leaves an empty file. Add/refresh contig lines from the reference.
        #  * norm: left-align, split multiallelics, check against the reference.
        #  * sort: the varseek records are not grouped by chromosome, so tabix
        #    indexing later fails with "chromosome blocks not continuous".
        reheader_proc = subprocess.Popen(
            ["bcftools", "reheader", "--fai", reference_fasta_index, test_vcf_unnormalized],
            stdout=subprocess.PIPE,
        )
        norm_proc = subprocess.Popen(
            ["bcftools", "norm", "-c", "w", "-f", reference_fasta, "-m", "-both", "-Ou"],
            stdin=reheader_proc.stdout, stdout=subprocess.PIPE,
        )
        reheader_proc.stdout.close()
        subprocess.run(["bcftools", "sort", output_type, "-o", test_vcf], stdin=norm_proc.stdout, check=True)
        norm_proc.stdout.close()
        if reheader_proc.wait() != 0:
            raise subprocess.CalledProcessError(reheader_proc.returncode, "bcftools reheader")
        if norm_proc.wait() != 0:
            raise subprocess.CalledProcessError(norm_proc.returncode, "bcftools norm")

    # sites-only VCFs (e.g. varseek) need a placeholder genotype for hap.py
    ensure_genotype_column(test_vcf)

    if test_vcf.endswith(".gz") and not os.path.isfile(f"{test_vcf}.tbi"):
        subprocess.run(["bcftools", "index", "-f", "-t", test_vcf], check=True)

    return test_vcf

def compare_with_happy(varseek_vcf, alternate_vcf, reference_fasta, output_dir, output_prefix="happy"):
    varseek_vcf = os.path.abspath(varseek_vcf)
    alternate_vcf = os.path.abspath(alternate_vcf)
    reference_fasta = os.path.abspath(reference_fasta)
    output_dir = os.path.abspath(output_dir)
    
    varseek_vcf_dir = os.path.dirname(varseek_vcf)
    alternate_vcf_dir = os.path.dirname(alternate_vcf)
    reference_fasta_dir = os.path.dirname(reference_fasta)

    reference_fasta_index = f"{reference_fasta}.fai"
    if not os.path.isfile(reference_fasta_index):
        subprocess.run(["samtools", "faidx", reference_fasta], check=True)

    # Drop non-variant (hom-ref / no-call) records from the caller VCF before
    # comparison; varseek emits only variant sites so it needs no such filter.
    alternate_vcf = filter_to_variant_calls(alternate_vcf)

    if not is_vcf_normalized(alternate_vcf):
        alternate_vcf = make_normalized_vcf(alternate_vcf, reference_fasta)

    if not is_vcf_normalized(varseek_vcf):
        varseek_vcf = make_normalized_vcf(varseek_vcf, reference_fasta)

    summary_csv_path = os.path.join(output_dir, f"{output_prefix}.summary.csv")

    if os.path.isfile(summary_csv_path):
        print(f"Summary file {summary_csv_path} already exists. Skipping hap.py run.")
    else:
        os.makedirs(output_dir, exist_ok=True)
        output_prefix_full = os.path.join(output_dir, output_prefix)
        command = f"podman run --rm -v {varseek_vcf_dir}:{varseek_vcf_dir} -v {alternate_vcf_dir}:{alternate_vcf_dir} -v {reference_fasta_dir}:{reference_fasta_dir} -v {output_dir}:{output_dir} mgibio/hap.py:v0.3.12 /opt/hap.py/bin/hap.py -r {reference_fasta} --engine=scmp-somatic -o {output_prefix_full} {varseek_vcf} {alternate_vcf}"
        subprocess.run(command, shell=True, check=True)


time: 28.9 ms (started: 2026-07-16 15:44:09 -07:00)


# de novo

In [ ]:
fastq_r2_files = sorted(
    os.path.join(dirpath, file)
    for dirpath, _, files in os.walk(fastqs_dir)
    for file in files
    if file.endswith((".fastq.gz", ".fq.gz", ".fastq", ".fq")) and "_R2_" in file
)

if not os.path.exists(variants):
    vk.denovo(
        inputs=fastq_r2_files,
        sequences=sequences,
        gtf=gtf,
        aligner="STAR",
        star_genome_index_dir=star_genome_index_dir,
        star_alignment_prefix=f"{denovo_star_alignment_dir}/star_",
        out_bam_dir=denovo_bam_dir,
        output=variants_vcf,
        output_tsv=variants,
        min_counts=min_counts_denovo,
        threads=threads,
        read_length=91,
        verbose=1,
        technology=technology
    )

# Restrict the de novo calls to those also present in the COSMIC census, keyed on the
# `seq_id:variant` HGVS string (identical to COSMIC's HGVSG). This makes the de novo
# pipeline a strict subset of the COSMIC variant universe, which is what lets its VCRS
# reference be a subset of the COSMIC reference (checked in the comparison section).
os.makedirs(os.path.dirname(variants_cosmic_only), exist_ok=True)
if not os.path.exists(variants_cosmic_only):
    all_variants_df = pd.read_csv(variants, sep="\t", dtype=str)
    cosmic_keys = set(
        pd.read_csv(cosmic_tsv, sep="\t", usecols=["HGVSG"], dtype=str, low_memory=False)
        .dropna()["HGVSG"]
    )
    keys = all_variants_df["seq_id"].astype(str) + ":" + all_variants_df["variant"].astype(str)
    all_variants_df[keys.isin(cosmic_keys)].to_csv(variants_cosmic_only, sep="\t", index=False)
    print(f"de novo variants total: {len(all_variants_df):,}; also in COSMIC: {keys.isin(cosmic_keys).sum():,}")

variants_df = pd.read_csv(variants_cosmic_only, sep="\t")
print(f"Number of de novo (COSMIC-only) variants: {len(variants_df)}")
variants_df.head()

Number of de novo (COSMIC-only) variants: 1243


,seq_id,variant
0,1,g.2238792C>T
1,1,g.2288631T>A
2,1,g.2556714A>G
3,1,g.2563346G>A
4,1,g.7114297G>A


time: 76 ms (started: 2026-07-16 15:44:09 -07:00)


In [ ]:
if not os.path.exists(vcrs_index_denovo):
    vk_ref_output_dict_denovo = vk.ref(
        variants=variants_cosmic_only,
        sequences=sequences,
        seq_id_column="seq_id",
        var_column="variant",
        out=vk_ref_out_dir_denovo,
        reference_out_dir=reference_dir,
        vcrs_unfiltered_fasta_out=os.path.join(vk_ref_out_dir_denovo, "vcrs_unfiltered.fa"),
        index_out=vcrs_index_denovo,
        vcrs_t2g_out=vcrs_t2g_denovo,
        w=w,
        k=k,
        species="human",
    )

15:44:09 - INFO - Using COSMIC email from COSMIC_EMAIL environment variable
15:44:09 - INFO - Using COSMIC password from COSMIC_PASSWORD environment variable


15:44:51 - INFO - Using the seq_id_column:var_column 'seq_id:variant' columns as the variant header column.
15:44:51 - INFO - Removing 0 duplications > k
15:44:51 - INFO - Unfiltered (pre-filter, pre-merge) VCRS fasta written to data/cosmic_vs_denovo/vk_ref_denovo/vcrs_unfiltered.fa.
/home/jrich/Desktop/varseek-fresh/varseek/varseek_build.py:1218: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mutations["vcrs_sequence_kmer_length"] = mutations["vcrs_sequence"].apply(lambda x: len(x) if pd.notna(x) else 0)
15:44:51 - INFO - Removed 0 variant-containing reference sequences with length less than 41...
15:44:51 - INFO - Removed 0 variant-containing reference sequences containing more than 0 'N's...
15:44:51 - INFO - Downloading prebuilt bowtie2 index 

File downloaded successfully to data/reference/genome_bowtie2_index/genome_bowtie2_index.tar.gz


15:46:28 - INFO - Downloading prebuilt bowtie2 index for species=human, component=cdna


File downloaded successfully to data/reference/cdna_bowtie2_index/cdna_bowtie2_index.tar.gz


15:46:49 - INFO - Running bowtie2 alignment
Scanning bowtie2 alignments: 19690it [00:00, 1141555.68it/s]
15:46:55 - INFO - reference_type: genome_or_transcriptome (bowtie2, ref 1/2 = genome: data/reference/genome_bowtie2_index/bowtie_index_genome/ref): 666 / 1239 VCRSs matched a k-mer.
15:46:55 - INFO - Running bowtie2 alignment
Scanning bowtie2 alignments: 6261it [00:00, 1116946.85it/s]
15:46:57 - INFO - reference_type: genome_or_transcriptome (bowtie2, ref 2/2 = cdna: data/reference/cdna_bowtie2_index/bowtie_index_cdna/ref): 291 / 1239 VCRSs matched a k-mer.
15:46:57 - INFO - reference_type: genome_or_transcriptome (bowtie2), union across references: 667 / 1239 reads matched.
15:46:57 - INFO - After pseudoalignment (genome_or_transcriptome), 572 / 1239 VCRSs remain (46.17%); removed 667 that aligned to the reference.
15:46:57 - INFO - Removed 667 variant-containing reference sequences that pseudoaligned to the genome_or_transcriptome reference...
15:46:57 - WARNING - 
        572 var

time: 2min 55s (started: 2026-07-16 15:44:09 -07:00)


In [14]:
adata_cleaned_denovo = os.path.join(vk_count_out_dir_denovo, "adata_cleaned.h5ad")
if not os.path.exists(adata_cleaned_denovo):
    vk.count(
        fastqs_dir,
        index=vcrs_index_denovo,
        t2g=vcrs_t2g_denovo,
        technology=technology,
        out=vk_count_out_dir_denovo,
        threads=threads,
        strand="unstranded",
        min_counts=min_counts_clean,
    )

15:47:05 - INFO - Removing index files from fastq files list, as they are not utilized in kb count with technology 10XV3
15:47:05 - INFO - Setting length_required to 59 if fastqpp is run
15:47:05 - INFO - Skipping vk fastqpp because there was no use for it
15:47:05 - INFO - Running kb count with command: kb count -t 16 -k 59 -i data/cosmic_vs_denovo/vk_ref_denovo/vcrs_index_denovo.idx -g data/cosmic_vs_denovo/vk_ref_denovo/vcrs_t2g_denovo.txt -x 10XV3 --h5ad -o data/cosmic_vs_denovo/vk_count_denovo/kb_count_out_vcrs --overwrite --strand unstranded --num --mm --union data/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L001_R1_001.fastq.gz data/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L001_R2_001.fastq.gz data/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L002_R1_001.fastq.gz data/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L002_R2_001.fastq.gz
[2026-07-16 15:47:10,534]    INFO [count] Using index data/cosmic_vs_denovo/vk_ref_denovo/vcrs_index_denovo.idx to generate BUS file to data/cosmic_vs_denovo/vk_count_denovo/kb_count_out_vcrs fro

time: 2min 30s (started: 2026-07-16 15:47:05 -07:00)


In [15]:
adata_to_vcf(adata_cleaned_denovo, vk_count_variants_vcf_denovo, min_total_reads=min_total_reads_denovo)

Wrote 572 variants to data/cosmic_vs_denovo/vk_count_denovo/varseek_variants.vcf
time: 59.7 ms (started: 2026-07-16 15:49:35 -07:00)


# cosmic

In [16]:
cosmic_tsv = "data/cosmic/Cosmic_MutantCensus_v104_GRCh38.tsv"

if not os.path.exists(cosmic_tsv):
    raise FileNotFoundError(f"Cosmic TSV not found at {cosmic_tsv}; please download from https://cancer.sanger.ac.uk/cosmic/download/cosmic/v104/mutantcensus.")

cosmic_tsv_cols = pd.read_csv(cosmic_tsv, sep="\t", nrows=0).columns
if "seq_id" not in cosmic_tsv_cols or "variant" not in cosmic_tsv_cols:
    cosmic_df = pd.read_csv(cosmic_tsv, sep="\t")
    cosmic_df[["seq_id", "variant"]] = cosmic_df["HGVSG"].str.split(":", n=1, expand=True)
    cosmic_df.to_csv(cosmic_tsv, sep="\t", index=False)

time: 7.24 ms (started: 2026-07-16 15:49:35 -07:00)


In [19]:
if not os.path.exists(vcrs_index_cosmic):
    vk_ref_output_dict_cosmic = vk.ref(
        variants=cosmic_tsv,
        sequences=sequences,
        seq_id_column="seq_id",
        var_column="variant",
        out=vk_ref_out_dir_cosmic,
        reference_out_dir=reference_dir,
        vcrs_unfiltered_fasta_out=os.path.join(vk_ref_out_dir_cosmic, "vcrs_unfiltered.fa"),
        index_out=vcrs_index_cosmic,
        vcrs_t2g_out=vcrs_t2g_cosmic,
        w=w,
        k=k,
        species="human",
    )

15:55:17 - INFO - Using COSMIC email from COSMIC_EMAIL environment variable
15:55:17 - INFO - Using COSMIC password from COSMIC_PASSWORD environment variable
/home/jrich/Desktop/varseek-fresh/varseek/varseek_build.py:747: DtypeWarning: Columns (14,26) have mixed types. Specify dtype option on import or set low_memory=False.
  if var_id_column:
15:56:41 - INFO - Using the seq_id_column:var_column 'seq_id:variant' columns as the variant header column.
15:57:07 - INFO - Removing 110 duplications > k
15:58:32 - INFO - Unfiltered (pre-filter, pre-merge) VCRS fasta written to data/cosmic_vs_denovo/vk_ref_cosmic/vcrs_unfiltered.fa.
/home/jrich/Desktop/varseek-fresh/varseek/varseek_build.py:1218: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # Get number

time: 35min 18s (started: 2026-07-16 15:55:17 -07:00)


In [20]:
adata_cleaned_cosmic = os.path.join(vk_count_out_dir_cosmic, "adata_cleaned.h5ad")
if not os.path.exists(adata_cleaned_cosmic):
    vk.count(
        fastqs_dir,
        index=vcrs_index_cosmic,
        t2g=vcrs_t2g_cosmic,
        technology=technology,
        out=vk_count_out_dir_cosmic,
        threads=threads,
        strand="unstranded",
        min_counts=min_counts_clean,
    )

16:30:36 - INFO - Removing index files from fastq files list, as they are not utilized in kb count with technology 10XV3
16:30:36 - INFO - Setting length_required to 59 if fastqpp is run
16:30:36 - INFO - Skipping vk fastqpp because there was no use for it
16:30:36 - INFO - Running kb count with command: kb count -t 16 -k 59 -i data/cosmic_vs_denovo/vk_ref_cosmic/vcrs_index_cosmic.idx -g data/cosmic_vs_denovo/vk_ref_cosmic/vcrs_t2g_cosmic.txt -x 10XV3 --h5ad -o data/cosmic_vs_denovo/vk_count_cosmic/kb_count_out_vcrs --overwrite --strand unstranded --num --mm --union data/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L001_R1_001.fastq.gz data/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L001_R2_001.fastq.gz data/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L002_R1_001.fastq.gz data/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L002_R2_001.fastq.gz
[2026-07-16 16:30:41,579]    INFO [count] Using index data/cosmic_vs_denovo/vk_ref_cosmic/vcrs_index_cosmic.idx to generate BUS file to data/cosmic_vs_denovo/vk_count_cosmic/kb_count_out_vcrs fro

time: 3min 58s (started: 2026-07-16 16:30:36 -07:00)


In [ ]:
adata_to_vcf(adata_cleaned_cosmic, vk_count_variants_vcf_cosmic, min_total_reads=min_total_reads_cosmic)

data/varseek_count_out_pbmc_1k_v3_cosmic/varseek_variants.vcf already exists, skipping VCF writing.
time: 304 μs (started: 2026-07-09 16:24:15 -07:00)


# comparison

In [ ]:
# Subset check: because a VCRS is fully determined by (variant, w, k, genome), the de novo
# (COSMIC-only) reference should be a subset of the full COSMIC reference. We compare the
# generated FASTAs at the sequence level. varseek collapses reverse-complement-identical
# VCRSs (unstranded), so we compare canonical (min of seq / its rev-comp) sequences.
_COMP = str.maketrans("ACGT", "TGCA")
def _rc(s):
    return s.translate(_COMP)[::-1]
def _canonical_seqs(fasta_path):
    seqs, cur = set(), []
    with open(fasta_path) as f:
        for line in f:
            if line.startswith(">"):
                if cur:
                    s = "".join(cur); seqs.add(min(s, _rc(s))); cur = []
            else:
                cur.append(line.strip())
    if cur:
        s = "".join(cur); seqs.add(min(s, _rc(s)))
    return seqs

def check_subset(denovo_fa, cosmic_fa, label):
    den = _canonical_seqs(denovo_fa)
    cos = _canonical_seqs(cosmic_fa)
    missing = den - cos
    status = "TRUE (subset)" if not missing else f"FALSE ({len(missing)} not in COSMIC)"
    print(f"{label:16s}: de novo {len(den):>7,} VCRS | COSMIC {len(cos):>9,} VCRS | subset -> {status}")
    return missing

print("de novo (COSMIC-only) VCRS reference  ⊆  full COSMIC VCRS reference?\n")
_miss_raw = check_subset(
    os.path.join(vk_ref_out_dir_denovo, "vcrs_unfiltered.fa"),
    os.path.join(vk_ref_out_dir_cosmic, "vcrs_unfiltered.fa"),
    "vcrs_unfiltered.fa",
)
_miss_filt = check_subset(
    os.path.join(vk_ref_out_dir_denovo, "vcrs.fa"),
    os.path.join(vk_ref_out_dir_cosmic, "vcrs.fa"),
    "vcrs.fa",
)
assert not _miss_raw, "de novo vcrs_unfiltered.fa is NOT a subset of the COSMIC vcrs_unfiltered.fa"
print("\nSubset property holds: every de novo VCRS also appears in the COSMIC reference.")

de novo (COSMIC-only) VCRS reference  ⊆  full COSMIC VCRS reference?



vcrs.fa         : de novo   1,239 VCRS | COSMIC 1,451,970 VCRS | subset -> TRUE (subset)


vcrs_filtered.fa: de novo     548 VCRS | COSMIC 1,328,106 VCRS | subset -> TRUE (subset)

Subset property holds: every de novo VCRS also appears in the COSMIC reference.
time: 3.93 s (started: 2026-07-09 16:24:15 -07:00)


In [ ]:
happy_out = os.path.join(root, "happy_output")
label = "varseek_cosmic"

# Truth = de novo (COSMIC-only) calls, query = full COSMIC pipeline calls.
compare_with_happy(
    varseek_vcf=vk_count_variants_vcf_denovo,
    alternate_vcf=vk_count_variants_vcf_cosmic,
    reference_fasta=sequences,
    output_dir=happy_out,
    output_prefix=f"varseek_vs_{label}"
)

Normalized file /home/jrich/Desktop/varseek-examples/data/varseek_count_out_pbmc_1k_v3_cosmic/varseek_variants_normalized.vcf already exists. Skipping normalization.


Writing to /home/jrich/tmp/bcftools.jxCTgi
Lines   total/split/joined/realigned/removed/skipped:	444/0/0/0/0/0
Merging 1 temporary files
Done
Cleaning


2026-07-09 23:24:20,273 WARNING  No reference file found at default locations. You can set the environment variable 'HGREF' or 'HG19' to point to a suitable Fasta file.


[I] Total VCF records:         444
[I] Non-reference VCF records: 444
[W] overlapping records at 1:204557929 for sample 0
[W] Variants that overlap on the reference allele: 12
[I] Total VCF records:         1745
[I] Non-reference VCF records: 1745


Hap.py v0.3.12


Benchmarking Summary:
Type Filter  TRUTH.TOTAL  TRUTH.TP  TRUTH.FN  QUERY.TOTAL  QUERY.FP  QUERY.UNK  FP.gt  FP.al  METRIC.Recall  METRIC.Precision  METRIC.Frac_NA  METRIC.F1_Score  TRUTH.TOTAL.TiTv_ratio  QUERY.TOTAL.TiTv_ratio  TRUTH.TOTAL.het_hom_ratio  QUERY.TOTAL.het_hom_ratio
INDEL    ALL            3         3         0          237       234          0      0      0       1.000000          0.012658             0.0         0.025000                     NaN                     NaN                        NaN                        NaN
INDEL   PASS            3         3         0          237       234          0      0      0       1.000000          0.012658             0.0         0.025000                     NaN                     NaN                        NaN                        NaN
  SNP    ALL          441       348        93         1508      1160          0      0      0       0.789116          0.230769             0.0         0.357106                2.218978          

time: 8.99 s (started: 2026-07-09 16:24:19 -07:00)


In [ ]:
# Collect the hap.py summary tables into one view.
# varseek is the TRUTH set and each caller is the QUERY, so:
#   Recall    = fraction of varseek calls confirmed by the caller
#   Precision = fraction of the caller's calls also in varseek (low by design:
#               varseek reports a small targeted subset vs. a genome-wide caller)
summary_frames = []
for label, alternate_vcf in [("varseek_cosmic", vk_count_variants_vcf_cosmic)]:
    summary_csv_path = os.path.join(happy_out, f"varseek_vs_{label}.summary.csv")
    if not os.path.isfile(summary_csv_path):
        continue
    df = pd.read_csv(summary_csv_path)
    df.insert(0, "Caller", label)
    summary_frames.append(df)

if summary_frames:
    summary_all = pd.concat(summary_frames, ignore_index=True)
    cols = ["Caller", "Type", "Filter", "TRUTH.TOTAL", "TRUTH.TP", "TRUTH.FN",
            "QUERY.TOTAL", "QUERY.FP", "METRIC.Recall", "METRIC.Precision", "METRIC.F1_Score"]
    display(summary_all[cols])
else:
    print("No hap.py summaries found yet -- run the comparison cell above first.")


,Caller,Type,Filter,TRUTH.TOTAL,TRUTH.TP,TRUTH.FN,QUERY.TOTAL,QUERY.FP,METRIC.Recall,METRIC.Precision,METRIC.F1_Score
0,varseek_cosmic,INDEL,ALL,3,3,0,237,234,1.000000,0.012658,0.025000
1,varseek_cosmic,INDEL,PASS,3,3,0,237,234,1.000000,0.012658,0.025000
2,varseek_cosmic,SNP,ALL,441,348,93,1508,1160,0.789116,0.230769,0.357106
3,varseek_cosmic,SNP,PASS,441,348,93,1508,1160,0.789116,0.230769,0.357106


time: 16.9 ms (started: 2026-07-09 16:24:28 -07:00)
